In [ ]:
import torch
import torchvision
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, ChainDataset, ConcatDataset
from torch.nn import functional as F
from torchvision.transforms.v2 import RandomCrop, Resize
from torchvision.transforms.v2 import functional as tvF
from torchgeo import models, datasets
import torchgeo

epochs = 50
batch_size = 48
device = 'cuda'
images_size = 512
data_root = 'workspace/data'

rand_crop = RandomCrop(size=images_size)
resize =  Resize(size = (images_size, images_size), interpolation = torchvision.transforms.InterpolationMode.BILINEAR, antialias=False)

def GeoNRW_new_mask(mask):
    mask = mask[0]
    
    roads = (mask==9)*1
    buildings = (mask==10)*2
    
    return roads + buildings


def GeoNRW_transform(sample):
    mask = GeoNRW_new_mask(sample['mask']);image = sample['image']
    size = image.shape[1]
    images_size_rand = torch.randint(low=1, high=3, size=(1,)).item() *256
    
    size_left = size - images_size_rand
    
    top = torch.randint(low=0, high=size_left, size=(1,)).item()
    left = torch.randint(low=0, high=size_left, size=(1,)).item()
    
    image = tvF.crop(inpt = image, top = top, left = left, height=images_size_rand, width=images_size_rand); image = resize(image)
    mask = tvF.crop(inpt = mask, top = top, left = left, height=images_size_rand, width=images_size_rand); mask = resize(mask[None, ...])[0]
    
    return {'image':image, 'mask':mask}


def LoveDA_new_mask(mask):
    roads = (mask==3)*1
    buildings = (mask==2)*2
    
    return roads + buildings

def LoveDA_transform(sample):
    mask = LoveDA_new_mask(sample['mask']); image = sample['image']/255.0
    size = image.shape[1]
    size_left = size - images_size
    
    top = torch.randint(low=0, high=size_left, size=(1,)).item()
    left = torch.randint(low=0, high=size_left, size=(1,)).item()
    
    image = tvF.crop(inpt = image, top = top, left = left, height=images_size, width=images_size)
    mask = tvF.crop(inpt = mask, top = top, left = left, height=images_size, width=images_size)

    return {'image' :image, 'mask':mask}




model = torchgeo.models.FarSeg(backbone='resnet50', classes=3, backbone_pretrained=True).to(device)
model = torch.compile(model)

# LoveDA 8 classes, 0 = background, 2 = building, 3 = road
# 2713 urban scene and 3274 rural
# images of size 1024x1024px unormalized [0,255.0]
train_set_LoveDA = torchgeo.datasets.LoveDA(root=data_root, split='train', scene=['rural', 'urban'], transforms=LoveDA_transform, download=True, checksum=False)
val_set_LoveDA = torchgeo.datasets.LoveDA(root=data_root, split='val', scene=['rural', 'urban'], transforms=LoveDA_transform, download=True, checksum=False)



# GeoNRW 11 classes, 0 = background, 9 = roads, 10 = buildings
# 7298 training and 485 test samples of 1000x1000px normalized to [0, 1]
train_set_GeoNRW = torchgeo.datasets.GeoNRW(root=data_root, split = 'train', transforms = GeoNRW_transform, download = True, checksum = False)
val_set_GeoNRW = torchgeo.datasets.GeoNRW(root=data_root, split='test', transforms=GeoNRW_transform, download=True, checksum=False)

# LandCoverAIBase 5 classes, 0=background, building = 1, road=4
# 33 orthophotos with 25 cm per pixel resolution (~9000x9500 px)
# 8 orthophotos with 50 cm per pixel resolution (~4200x4700 px)
# train_set_LandCover = torchgeo.datasets.LandCoverAIBase(root=data_root, download=True, checksum=False)

train_set= ConcatDataset([train_set_LoveDA, train_set_GeoNRW])
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=24, drop_last=True, prefetch_factor=2)

val_set = ConcatDataset([val_set_LoveDA, val_set_GeoNRW])
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=24, drop_last=True, prefetch_factor=2)

print(len(train_set_LoveDA), len(val_set_LoveDA))
print(len(train_set_GeoNRW), len(val_set_GeoNRW))
print(len(train_loader), len(val_loader))


In [ ]:
id = 0
x = train_set_GeoNRW[id]; y = train_set_LoveDA[id]
print(x['image'].shape, y['image'].shape, x['mask'].shape, y['mask'].sha)

In [ ]:
from matplotlib import pyplot as plt 

id = 240 
data = val_set_GeoNRW[id]

plt.figure(figsize=(20,20))
plt.subplot(1,2,1)
plt.imshow(data['image'].permute([1,2,0]))
plt.subplot(1,2,2)
plt.imshow(data['mask']*100, cmap='gray')
plt.show()


id = 600 
data = val_set_LoveDA[id]

plt.figure(figsize=(20,20))
plt.subplot(1,2,1)
plt.imshow(data['image'].permute([1,2,0]))
plt.subplot(1,2,2)
plt.imshow(data['mask']*100, cmap='gray')
plt.show()



In [ ]:
total_steps = epochs*len(train_loader)
optimizer = AdamW(model.parameters(), lr= 1e-5)
lr_scheduler = OneCycleLR(optimizer, max_lr= 1e-3, total_steps=total_steps, pct_start = 0.1)
scaler = torch.cuda.amp.GradScaler()

# print(len(data_loader))


def model_eval(model, val_loader):
    model.eval()
    val_loss = []
    for data in val_loader:
        image = data['image'].to(device); mask = data['mask'].to(device)

        with torch.no_grad():
            with torch.autocast(device_type=device, dtype=torch.float16):
                out = model(image)
                loss = F.cross_entropy(out, mask)
            val_loss.append(loss.item())
    model.train()
    return sum(val_loss)/len(val_loader)
    

import time

train_loss = []
val_loss = []
step = 0
steps_print = 50
for epoch in range(epochs):
    for data in train_loader:
        image = data['image'].to(device); mask = data['mask'].to(device)
        time_start = time.time()


        optimizer.zero_grad()
        with torch.autocast(device_type=device, dtype=torch.float16):
            out = model(image)
            loss = F.cross_entropy(out, mask)
        train_loss.append(loss.item())    
                
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()
        if step%steps_print == 0: 
            print_loss = sum(train_loss[-steps_print:])/steps_print
            print(f'epoch = {epoch} step = {step} lr = {lr_scheduler.get_last_lr()[0]:.5f} loss = {print_loss} time_per_step ={time.time()-time_start:.3f}')
        step+=1

    val_loss.append(model_eval(model, val_loader))
    print(f'£££££££££                                         £££££££££££')
    print(f'£££££££££       val_loss = {val_loss[-1]}         £££££££££££')
    print(f'£££££££££                                         £££££££££££')
    
torch.save(model.state_dict(), 'workspace/FarSeg.pth')


# # total_steps = 209550
# from matplotlib import pyplot as plt 
# new_loss = torch.tensor(training_loss[:209500]).view(100, -1).mean(dim=0)

# plt.plot(range(len(new_loss)), new_loss)
# plt.savefig('workspace/training_loss.png')